In [0]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, DoubleType, LongType, StringType
from pyspark.sql.functions import col, from_unixtime, to_timestamp

In [0]:
apiKey="XQf22C5alIB5vXvUqKsjb4f7BZIQIKaH"

In [0]:
pip install -U polygon-api-client

In [0]:
from polygon import RESTClient
import pandas as pd
from pyspark.sql.types import StructType, StructField, DoubleType, LongType, StringType
from pyspark.sql.functions import col, to_timestamp, from_unixtime
from pyspark.sql import SparkSession

def fetch_minute_bars_to_spark(spark, client, ticker, multiplier, start_date, end_date, lmt):
    # 1) Fetch aggregates (minute bars)
    aggs = []
    for a in client.list_aggs(
        ticker,
        multiplier,
        timespan,
        from_=start_date,
        to=end_date,
        limit=lmt
    ):
        aggs.append(a)

    # 2) Convert SDK objects to dicts with canonical Polygon keys
    # Polygon returns fields like o,h,l,c,v,vw,t,n,otc; SDK models may expose open/high/low/close/volume/vwap/timestamp/transactions
    aggs_dicts = []
    for a in aggs:
        d = a.__dict__ if hasattr(a, "__dict__") else (a._asdict() if hasattr(a, "_asdict") else dict(a))
        aggs_dicts.append({
            "open": d.get("open", d.get("o")),
            "high": d.get("high", d.get("h")),
            "low":  d.get("low",  d.get("l")),
            "close": d.get("close", d.get("c")),
            "volume": d.get("volume", d.get("v")),
            "vwap": d.get("vwap", d.get("vw")),
            "timestamp": d.get("timestamp", d.get("t")),
            "transactions": d.get("transactions", d.get("n")),
            # otc in raw JSON is boolean; your schema uses StringType, so cast to 'true'/'false' strings
            "otc": str(d.get("otc")).lower() if d.get("otc") is not None else None,
        })

    # 3) pandas DataFrame
    df_pd = pd.DataFrame(aggs_dicts)

    # 4) Spark schema (matches your request)
    schema = StructType([
        StructField("open", DoubleType(), True),
        StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True),
        StructField("close", DoubleType(), True),
        StructField("volume", LongType(), True),
        StructField("vwap", DoubleType(), True),
        StructField("timestamp", LongType(), True),
        StructField("transactions", LongType(), True),
        StructField("otc", StringType(), True),
    ])

    # 5) Set Spark session timezone to IST and create Spark DataFrame
    spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")
    spark_df = (
        spark.createDataFrame(df_pd, schema=schema)
             .withColumn("TimestampIst", (col("timestamp") / 1000).cast("double"))
             .withColumn("TimestampIst", to_timestamp(from_unixtime(col("TimestampIst"))))
    )

    return spark_df

client = RESTClient(api_key=apiKey)
ticker = "AAPL"
timespan = "minute"
multiplier = 1
start_date = "2025-09-19"
end_date = "2025-09-20"
lmt=50000
spark_df = fetch_minute_bars_to_spark(spark, client, ticker, multiplier, start_date, end_date, lmt)
spark_df.display()

In [0]:
import json

resp = client.get_aggs(
    ticker="AAPL",
    multiplier=1,
    timespan="month",
    from_="2025-09-19",
    to="2025-09-19",
    limit=50000,
    raw=True
)
data = json.loads(resp.data)
data

In [0]:
schema = "user_id long, device_id long, mac_address string, registration_timestamp double"
df_stream = (spark.readStream
                .format("cloudFiles")
                .schema(schema)
                .option("maxFilesPerTrigger", 1)
                .option("cloudFiles.format", "json")
                .option("header", "true")
                .withColumn("load_time", F.current_timestamp()) 
                .withColumn("source_file", F.input_file_name())
            )


stream_writer = df_stream.writeStream \
                            .format("delta") \
                            .option("checkpointLocation", "dbfs:/Volumes/dev/infra/pipeline_artifacts/checkpoint/aapl_minutes_data") \
                            .outputMode("append") \
                            .queryName("aapl_minutes_data")

stream_writer.trigger(processingTime=1 minutes).topath(f"dbfs:/Volumes/dev/infra/pipeline_artifacts/landing_zone")